In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ノイズ学習ヘルパー

<details>
<summary><b>Package versions</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
エラー緩和技術である [PEA](./error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) と [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) はいずれも、[Pauli-Lindbladノイズモデル](https://arxiv.org/abs/2201.09866)に基づくノイズ学習コンポーネントを使用しています。このコンポーネントは通常、`qiskit-ibm-runtime` を通じて1つ以上のジョブを送信した後の実行中に管理され、フィッティングされたノイズモデルへのローカルアクセスはありません。しかし、`qiskit-ibm-runtime` 0.27.1 以降、[`NoiseLearner`](../api/qiskit-ibm-runtime/noise-learner) クラスと関連する [`NoiseLearnerOptions`](../api/qiskit-ibm-runtime/options-noise-learner-options) クラスが作成され、これらのノイズ学習実験の結果を取得できるようになりました。これらの結果は `NoiseLearnerResult` としてローカルに保存し、後続の実験の入力として使用することができます。このページでは、その使用方法と利用可能なオプションの概要を説明します。

## 概要
`NoiseLearner` クラスは、1つ（または複数の）回路に対して、Pauli-Lindbladノイズモデルに基づいてノイズプロセスを特徴付ける実験を実行します。このクラスには `run()` メソッドがあり、学習実験を実行します。入力として回路のリストまたは [PUB](./primitive-input-output#overview-of-pubs) を受け取り、学習済みノイズチャネルと送信されたジョブに関するメタデータを含む `NoiseLearnerResult` を返します。以下は、ヘルパープログラムの使用方法を示すコードスニペットです。

In [1]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
from qiskit_ibm_runtime.noise_learner import NoiseLearner
from qiskit_ibm_runtime.options import (
    NoiseLearnerOptions,
    ResilienceOptionsV2,
    EstimatorOptions,
)

# Build a circuit with two entangling layers
num_qubits = 27
edges = list(CouplingMap.from_line(num_qubits, bidirectional=False))
even_edges = edges[::2]
odd_edges = edges[1::2]

circuit = QuantumCircuit(num_qubits)
for pair in even_edges:
    circuit.cx(pair[0], pair[1])
for pair in odd_edges:
    circuit.cx(pair[0], pair[1])

# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
circuit_to_learn = pm.run(circuit)

# Instantiate a NoiseLearner object and execute the noise learning program
learner = NoiseLearner(mode=backend)
job = learner.run([circuit_to_learn])
noise_model = job.result()

結果として得られる `NoiseLearnerResult.data` は、対象回路に属する個々のエンタングリング層ごとの[ノイズモデル](https://arxiv.org/abs/2201.09866)を含む [`LayerError`](../api/qiskit-ibm-runtime/utils-noise-learner-result-layer-error) オブジェクトのリストです。各 `LayerError` は、回路と量子ビットラベルのセットという形で層の情報を格納するとともに、該当する層に対して学習されたノイズモデルの [`PauliLindbladError`](../api/qiskit-ibm-runtime/utils-noise-learner-result-pauli-lindblad-error) も格納します。

In [2]:
print(
    f"Noise learner result contains {len(noise_model.data)} entries"
    f" and has the following type:\n {type(noise_model)}\n"
)
print(
    f"Each element of `NoiseLearnerResult` then contains"
    f" an object of type:\n {type(noise_model.data[0])}\n"
)
print(
    f"And each of these `LayerError` objects possess"
    f" data on the generators for the error channel: \n{noise_model.data[0].error.generators}\n"
)
print(f"Along with the error rates: \n{noise_model.data[0].error.rates}\n")

Noise learner result contains 2 entries and has the following type:
 <class 'qiskit_ibm_runtime.utils.noise_learner_result.NoiseLearnerResult'>

Each element of `NoiseLearnerResult` then contains an object of type:
 <class 'qiskit_ibm_runtime.utils.noise_learner_result.LayerError'>

And each of these `LayerError` objects possess data on the generators for the error channel: 
['IIIIIIIIIIIIIIIIIIIIIIIIIIX', 'IIIIIIIIIIIIIIIIIIIIIIIIIIY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIXI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIXX', 'IIIIIIIIIIIIIIIIIIIIIIIIIXY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIXZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIYI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIYX', 'IIIIIIIIIIIIIIIIIIIIIIIIIYY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIYZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIZI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIZX', 'IIIIIIIIIIIIIIIIIIIIIIIIIZY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIZZ', 'IIIIIIIIIIIIIIIIIIIIIIIIXII',
 'IIIIIIIIIIIIIIIIIIIIIIIIXXI', 'IIIIIIIIIIIIIIIIIIIIIIIIXYI',
 'IIIIIIIIIIIIIIIIIIIIIIIIXZI', 'IIIIIIIIIIIIIIIIIIIIII

The `LayerError.error` attribute of the noise learning result contains the generators and error rates of the fitted Pauli Lindblad model, which has the form

$$ \Lambda(\rho) = \exp{\sum_j r_j \left(P_j \rho P_j^\dagger - \rho\right)}, $$

where the $r_j$ are the `LayerError.rates` and $P_j$ are the Pauli operators specified in `LayerError.generators`.

## Noise learning options

You can choose among several options to input when you instantiate a `NoiseLearner` object. These options are encapsulated by the `qiskit_ibm_runtime.options.NoiseLearnerOptions` class and include the ability to specify the maximum layers to learn, number of randomizations, and the twirling strategy, among others. Refer to the API documentation about [`NoiseLearnerOptions`](../api/qiskit-ibm-runtime/options-noise-learner-options) for more detailed information.

Below is a simple example showing how to use the `NoiseLearnerOptions` in a `NoiseLearner` experiment:

In [3]:
# Build a GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))
# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
circuit_to_run = pm.run(circuit_to_learn)

# Instantiate a noise learner options object
learner_options = NoiseLearnerOptions(
    max_layers_to_learn=3, num_randomizations=32, twirling_strategy="all"
)

# Instantiate a NoiseLearner object and execute the noise learning program
learner = NoiseLearner(mode=backend, options=learner_options)
job = learner.run([circuit_to_run])
noise_model = job.result()

ノイズ学習結果の `LayerError.error` 属性には、フィッティングされた Pauli Lindblad モデルのジェネレーターとエラーレートが含まれています。このモデルは以下の形式を持ちます。

$$ \Lambda(\rho) = \exp{\sum_j r_j \left(P_j \rho P_j^\dagger - \rho\right)}, $$

ここで、$r_j$ は `LayerError.rates`、$P_j$ は `LayerError.generators` に指定された Pauli 演算子です。
## ノイズ学習オプション
`NoiseLearner` オブジェクトをインスタンス化する際に指定できるオプションがいくつかあります。これらのオプションは `qiskit_ibm_runtime.options.NoiseLearnerOptions` クラスにカプセル化されており、学習する最大層数、ランダム化の回数、twirling 戦略などを指定する機能が含まれています。詳細については、[`NoiseLearnerOptions`](../api/qiskit-ibm-runtime/options-noise-learner-options) の API ドキュメントを参照してください。

以下は、`NoiseLearner` 実験で `NoiseLearnerOptions` を使用する簡単な例です。

In [4]:
# pass the noise model to the `estimator.options` attribute directly
estimator = EstimatorV2(mode=backend)
estimator.options.resilience.layer_noise_model = noise_model

In [5]:
# Specify options via a ResilienceOptionsV2 object
resilience_options = ResilienceOptionsV2(layer_noise_model=noise_model)
estimator_options = EstimatorOptions(resilience=resilience_options)
estimator = EstimatorV2(mode=backend, options=estimator_options)

In [6]:
# Specify options via a dictionary
options_dict = {
    "resilience_level": 2,
    "resilience": {"layer_noise_model": noise_model},
}

estimator = EstimatorV2(mode=backend, options=options_dict)

Once the noise model is passed into the `EstimatorV2` object, it can be used to run workloads and perform error mitigation as normal.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Read more about [configuring error mitigation](configure-error-mitigation).
    - Review the [EstimatorOptions API reference](/docs/api/qiskit-ibm-runtime/options-estimator-options) and [ResilienceOptionsV2 API reference](/docs/api/qiskit-ibm-runtime/options-resilience-options-v2).
    - Learn more about [Error mitigation and suppression techniques](error-mitigation-and-suppression-techniques) that are available through Qiskit Runtime.
    - Review how to [Specify options](specify-runtime-options) for the Qiskit Runtime primitives.
    - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>